# Analyse trail — Récupérer les données météo de ton trail

Dans ce notebook qui fait suite au 3 précédents 
- lire les données
- analyser les données de terrain,
- afficher la trace d'une course ou d'un entrainement sur une carte

on s'intéresse à l'affichage la récupération des données météos


**Pré-requis :**

Cette fois, on va avoir besoin d'une librairie supplémentaire qui s'appelle `openmeteo_requests`


```bash
pip install fitparse pandas numpy matplotlib folium openmeteo_requests

```

**Remarque**

Dans les premiers notebooks, j'avais volontairement laissé le code complet dans les cellules. Cette fois, c'est fini, l'essentiel des codes est déplacé dans le fichier annexe `trail_analysis_pub.py`. Les notebooks qui contiennent de plus en plus de cellule d'analyses des données sont allégés. 

N'oublie pas de récupérer le fichier `trail_analysis_pub.py` en local pour faire tourner le notebook.

### 
### © Gregory Sainton
### 
- Notebook        : Analyse d'une séance de trail/course 
- Auteur          : Grégory Sainton <trinity4trail@proton.me>
- Dépôt           : https://github.com/gregs1t/trail
- Date de création: 2025-12
- Dernière modif. : 2026-04
- Version         : 1.0
---
- Licence         : CC BY-NC-SA 4.0
-                   https://creativecommons.org/licenses/by-nc-sa/4.0/

>    Vous êtes libre de partager et d'adapter ce travail, à condition de :
>     · citer l'auteur (BY)
>     · ne pas en faire un usage commercial (NC)
>     · redistribuer sous la même licence (SA)

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # trail_analysis gère ses propres filtres

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trail_analysis_pub import (
    load_fit, clean_df, plot_map,
    compute_slope, compute_dplus_dminus, compute_gap,
    segment_updown, classify_walk_run, plot_dashboard,
    plot_walk_by_slope_sections)

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
print("Imports OK.")

## Paramètres utilisateur

**Seule cellule à modifier.**

In [ ]:
TARGET = { 
         'label':      'macourse 2026',
         'fit_path':   'mon_fichier_a_moi.fit',
         'ravito_km':  [25.4, 37.7, 47.7, 53.8, 78.3],
         'ravito_nom': ['Buc', 'Chaville', 'Saint Philippe', 'Marcel Bec', 'Saint Cloud'],
    }

# --- Fichier FIT ---
FIT_PATH = TARGET["fit_path"]

# --- Physiologie ---
FC_MAX    = 453     # FC max réelle (bpm)
FC_MIN    = 22      # FC repos (bpm)
POIDS_KG  = 138.0    # masse corporelle (kg)

# --- Ravitaillements ---
RAVITO_KM  = TARGET["ravito_km"]
RAVITO_NOM = TARGET["ravito_nom"]

# --- Calcul de la pente ---
WINDOW_M     = 100.0    # fenêtre glissante en mètres
UP_THR       = +3.0     # seuil montée (%)
DOWN_THR     = -3.0     # seuil descente (%)
MIN_SEG_M    = 200.0    # longueur minimale de segment (m)

# --- Seuils marche / course ---
WALK_THR_KMH = 5.0      # seuil vitesse (km/h)
WALK_THR_CAD = 140.0    # seuil cadence (pas/min)

---
## 1. Chargement et nettoyage

In [ ]:
df_raw = load_fit(FIT_PATH)
df, ALT_COL = clean_df(df_raw)
df["slope_pct"] = compute_slope(df, WINDOW_M)
df = compute_gap(df)
df = segment_updown(df, UP_THR, DOWN_THR, MIN_SEG_M)

if "cadence" in df.columns:
    df = classify_walk_run(df, WALK_THR_KMH, WALK_THR_CAD)

dplus, dminus = compute_dplus_dminus(df)

print(f"Points       : {len(df)}")
print(f"Distance     : {df['dist_m'].max()/1000:.1f} km")
print(f"Durée        : {df['time_h'].max():.2f} h")
print(f"D+           : {dplus:.0f} m")
print(f"D-           : {dminus:.0f} m")
if "heart_rate" in df.columns:
    print(f"FC moy / max : {df['heart_rate'].mean():.0f} / {df['heart_rate'].max():.0f} bpm")

---
## 2. Cartographie de la trace

In [ ]:
# Carte colorée par FC (changer color_col selon besoin : "speed_kmh", "slope_pct", None)
# animated=True      : trace animée AntPath (sens de course)
# show_elevation=True : profil altimétrique en incrustation (coin inférieur gauche)
m = plot_map(
    df,
    ravito_km=RAVITO_KM,
    ravito_nom=RAVITO_NOM,
    color_col="heart_rate",
    cmap_name="RdYlGn_r",
    animated=True,
    show_elevation=True,
)
m   # affiche la carte interactive dans Jupyter

---
## 3. Profil altimétrique

In [ ]:
x = df["dist_m"] / 1000.0
y = df["alt_m"]

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(x, y, color="brown", linewidth=2)
ax.fill_between(x, y, y.min(), color="brown", alpha=0.3)

for km, nom in zip(RAVITO_KM, RAVITO_NOM):
    idx = (df["dist_m"] / 1000.0 - km).abs().idxmin()
    ax.axvline(km, color="grey", linestyle=":", alpha=0.6)
    ax.text(km, ax.get_ylim()[1] * 0.95, nom,
            fontsize=7, rotation=90, va="top", ha="right", color="grey")

ax.set_xlabel("Distance (km)")
ax.set_ylabel("Altitude (m)")
ax.set_title("Profil altimétrique + ravitaillements")
fig.tight_layout()
plt.show()

---
## 4. Dashboard FC / allure / température

In [ ]:
plot_dashboard(df, FC_MIN, FC_MAX)

---
## Marche par classe de pente

In [ ]:
SLOPE_BINS   = [-30, -10, -3, 3, 10, 20, 40]
SLOPE_LABELS = ["< -10%", "-10/-3%", "plat", "+3/+10%", "+10/+20%", "> +20%"]

if "cadence" in df.columns:
    plot_walk_by_slope_sections(
        df, slope_bins=SLOPE_BINS, slope_labels=SLOPE_LABELS,
        ravito_km=RAVITO_KM,
    )
else:
    print("Pas de cadence.")

# Et si on veut une version avec des segments identiques :
# Par exemple, 2 segments pour distinguer les deux moitiés de la course.
if "cadence" in df.columns:
    plot_walk_by_slope_sections(
        df, slope_bins=SLOPE_BINS, slope_labels=SLOPE_LABELS,
        n_segments=2)
else:
    print("Pas de cadence.")

## Météo — Température, humidité, vent

> Les données sont récupérées depuis l'API **Open-Meteo Historical Weather**
> (`archive-api.open-meteo.com`) via le SDK `openmeteo-requests` (FlatBuffers).
> Deux modèles disponibles : **ERA5-Land** (0.1°, ~9 km) et **ERA5** (0.25°, ~25 km).
>
> **Course nocturne franchissant minuit** : la requête couvre automatiquement
> les deux jours (start_date → end_date extraits du FIT).
>
> **Température montre** : masquée par défaut — biaisée +2 à +5°C
> par la chaleur corporelle. Activer `show_watch_temp=True` pour comparaison.

In [ ]:
from trail_analysis_pub import (
    fetch_weather_hourly,
    enrich_df_with_weather,
    plot_weather_along_race,
)

# --- Paramètres météo (extraits automatiquement du DataFrame) ---

# Point de référence : centroïde GPS de la course
if "lat" in df.columns and "lon" in df.columns:
    REF_LAT = float(df["lat"].dropna().median())
    REF_LON = float(df["lon"].dropna().median())
else:
    REF_LAT = 45.53   # ← adapter si coordonnées GPS absentes
    REF_LON =  4.45

# Date de début et de fin extraites des timestamps FIT
# Important pour les courses nocturnes qui franchissent minuit :
# la SaintéLyon démarre le 29/11 à 23h31 et finit le 30/11 à 10h
from datetime import timedelta
DATE_START = df["timestamp"].iloc[0]
DATE_END   = df["timestamp"].iloc[-1]

DATE_STR_START = DATE_START.strftime("%Y-%m-%d")
DATE_STR_END   = DATE_END.strftime("%Y-%m-%d")

# Fuseau horaire IANA
TIMEZONE = "Europe/Paris"

print(f"Référence GPS    : lat={REF_LAT:.4f}°  lon={REF_LON:.4f}°")
print(f"Début course     : {DATE_START}")
print(f"Fin course       : {DATE_END}")
print(f"Dates requête    : {DATE_STR_START} → {DATE_STR_END}")
print(f"Fuseau horaire   : {TIMEZONE}")


In [ ]:
# --- Récupération ERA5-Land (SDK openmeteo-requests) ---

# Modèle de réanalyse :
#   "era5_land" : résolution ~9 km  — précis en altitude, peut sous-estimer
#                 en zone urbaine dense (typique : départ SaintéLyon, Lyon)
#   "era5"      : résolution ~25 km — parfois plus juste en plaine/péri-urbain
METEO_MODEL = "era5_land"   # ← "era5" pour tester l'alternative

print(f"Récupération données {METEO_MODEL.upper()} via Open-Meteo...")
df_weather = fetch_weather_hourly(
    lat=REF_LAT,
    lon=REF_LON,
    date_str=DATE_STR_START,
    date_end=DATE_STR_END,      # couvre les courses qui franchissent minuit
    timezone=TIMEZONE,
    model=METEO_MODEL,
)

if df_weather is not None:
    print(f"  → {len(df_weather)} heures récupérées ({METEO_MODEL})")
    # cols_show = ["time", "temperature_2m", "relative_humidity_2m",
    #              "wind_speed_10m", "precipitation", "shortwave_radiation"]
    # print(df_weather[cols_show].to_string(index=False))
else:
    print("⚠️  Données météo indisponibles.")


# --- Interpolation au pas seconde sur le DataFrame GPS ---
df = enrich_df_with_weather(df, df_weather)

meteo_cols = [c for c in ["temp_api", "humidity_api", "wind_kmh_api",
                           "solar_rad_api", "wbgt_api"]
              if c in df.columns]
# if meteo_cols:
#     print(f"\nColonnes météo ajoutées : {meteo_cols}")
#     print(df[meteo_cols].describe().round(1).to_string())

# --- Visualisation ---
# show_watch_temp=True pour voir la courbe montre en comparaison
plot_weather_along_race(df, RAVITO_KM, RAVITO_NOM, fc_max=FC_MAX,
                        show_watch_temp=False)
